In [ ]:
%pip install sdv pandas numpy plt seaborn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sdv.datasets.demo import download_demo
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from rdt.transformers.numerical import GaussianNormalizer, ClusterBasedNormalizer, FloatFormatter
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Cargar el dataset de ejemplo oficial
# Este dataset está pre-limpiado para pruebas de síntesis
url = "../../datasources/bank_full_dataset/bank-full.csv"
df_real = pd.read_csv(url, sep=';')
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_real)

print("Columnas del dataset:", df_real.columns.tolist())
df_real.head()

num_cols = df_real.select_dtypes(include=[np.number]).columns
print("Columnas numericas:", num_cols)
cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
print("Columnas categoricas:", cat_cols)


Columnas del dataset: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Columnas numericas: Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')
Columnas categoricas: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')


In [8]:
import time
from sdv.utils import load_synthesizer

IMPORT_PICKLE=True

# Configurar el sintetizador por TVAE

if IMPORT_PICKLE:
    synthesizer = load_synthesizer(filepath='ctgan_64ep_250bs.pkl')

    print("Sintetizador cargado desde archivo pickle!")

else:
    synthesizer = CTGANSynthesizer(
        metadata,
        generator_dim=(512, 512),       # Red más ancha para "memoria" de correlación
        discriminator_dim=(512, 512),   # Discriminador más fuerte para detectar el "gap"
        pac=10,                         # Agrupación de muestras para evitar mode collapse
        discriminator_steps=5,          # Entrenar más el discriminador por cada paso
        epochs=64,                     # Con redes más grandes, necesitamos más épocas
        batch_size=250,
        log_frequency=True,
        enforce_rounding=True,
        verbose=True
    )

    # Tratamiento de precisión para pasar el Test K-S
    # El GaussianNormalizer es más estricto y fiel para distribuciones continuas
    synthesizer.auto_assign_transformers(df_real)
    
    synthesizer.update_transformers(column_name_to_transformer={
        'balance': ClusterBasedNormalizer(model_missing_values=True),
        'duration': GaussianNormalizer(model_missing_values=True),
        'day': ClusterBasedNormalizer(model_missing_values=True, enforce_min_max_values=True, max_clusters=31),
        'pdays': ClusterBasedNormalizer(model_missing_values=True),
    })
    
    print("Entrenando modelo CTGANSynthesizer ...")
    start_time = time.perf_counter()

    synthesizer.fit(df_real)

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time


    synthesizer.save("ctgan_64ep_250bs.pkl")

    print(f"Entrenamiento terminado en {elapsed_time:.4f} segundos.")

df_synth = synthesizer.sample(num_rows=len(df_real))
df_synth['age'] = df_synth['age'].round().astype(int)
df_synth['day'] = df_synth['day'].round().astype(int)

df_synth.to_csv('ctgan_synth_out.csv')
print(f"Se han generado {len(df_real)} nuevos registros sinteticos.")


Sintetizador cargado desde archivo pickle!
Se han generado 45211 nuevos registros sinteticos.
